#1. Inspect the table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
orders_df = spark.table(
    "adb_ecommerce_de_7405607562556202.ecommerce_bronze.orders"
)

orders_df.printSchema()
orders_df.show(10, truncate=False)

### Check for duplicates


In [0]:
orders_df.groupBy("order_id").count().filter("count > 1").show()

###Check for nulls

In [0]:
orders_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in orders_df.columns]).show()

#2. Silver Tranformations

In [0]:
#Trim order_status column and remove duplicates from order_id

silver_orders = orders_df.withColumn('order_status', lower(trim(col("order_status"))))\
    .dropDuplicates(['order_id'])

In [0]:
#make new col for purchase date containing only date
silver_orders = silver_orders.withColumn('order_purchase_date', to_date(col('order_purchase_timestamp')))

In [0]:
#rename all the columns

RENAME = {
    'order_purchase_timestamp': 'purchase_timestamp',
    'order_approved_at': 'approved_timestamp',
    'order_delivered_carrier_date': 'delivered_carrier_timestamp',
    'order_delivered_customer_date': 'delivered_customer_timestamp',
    'order_estimated_delivery_date': 'estimated_delivery_timestamp',
    'order_purchase_date': 'purchase_date'
}

for old, new in RENAME.items():
    silver_orders = silver_orders.withColumnRenamed(old, new)


In [0]:
silver_orders.show(5)

#3. Write silver to ADLS

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/orders"

silver_orders.write.format("delta").mode("overwrite").save(silver_path)


#4. Register Silver in Unity Catalog

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS adb_ecommerce_de_7405607562556202.ecommerce_silver.orders
USING DELTA
LOCATION '{silver_path}'
""")


#silver_orders.write \
#     .mode("overwrite") \
#     .format("delta") \
#     .saveAsTable(
#         "adb_ecommerce_de_7405607562556202.ecommerce_silver.orders"
#     )

In [0]:
%sql
select * from adb_ecommerce_de_7405607562556202.ecommerce_silver.orders limit 5